In [ ]:
import numpy as np
import json
from pathlib import Path

import torch
from torch.utils.data import Dataset
from transformers import (
    GPT2LMHeadModel, GPT2Config, PreTrainedTokenizerFast,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
)

# ── Paths ────────────────────────────────────────────────────────────────
TOKEN_DIR  = Path('../dataset/tokenized_hierarchical/test')   # switch to 'full' for production
OUTPUT_DIR = Path('../models/distilgpt2_53tet')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1 — Load Tokenized Data

In [ ]:
token_ids = np.load(TOKEN_DIR / 'tokens.npy')    # int32
offsets   = np.load(TOKEN_DIR / 'offsets.npy')    # int64
vocab     = json.loads((TOKEN_DIR / 'vocab.json').read_text())

tok2id = vocab['tok2id']
id2tok = {int(k): v for k, v in vocab['id2tok'].items()}
VOCAB_SIZE = len(tok2id)

print(f'Tokens:     {len(token_ids):>10,}')
print(f'Songs:      {len(offsets)-1:>10,}')
print(f'Vocab size: {VOCAB_SIZE:>10,}')
print(f'\nSample tokens: {[id2tok[i] for i in token_ids[:20]]}')

## 2 — Sliding-Window Dataset

Extracts overlapping windows of `block_size` from the flat token array.
Songs are concatenated — the `<START>` / `<END>` tokens provide natural boundaries.

In [ ]:
class TokenWindowDataset(Dataset):
    """Sliding-window dataset over a flat int32 token array."""

    def __init__(self, token_ids: np.ndarray, block_size: int = 512, stride: int = 256):
        self.data = torch.from_numpy(token_ids.astype(np.int64))
        self.block_size = block_size
        self.stride = stride
        self.n_windows = max(1, (len(self.data) - block_size) // stride)

    def __len__(self):
        return self.n_windows

    def __getitem__(self, idx):
        start = idx * self.stride
        chunk = self.data[start : start + self.block_size]
        return {'input_ids': chunk, 'labels': chunk.clone()}


BLOCK_SIZE = 512
STRIDE     = 256

# Train / val split (90/10 by songs)
n_songs   = len(offsets) - 1
split_idx = int(n_songs * 0.9)
train_end = int(offsets[split_idx])

train_ds = TokenWindowDataset(token_ids[:train_end], BLOCK_SIZE, STRIDE)
val_ds   = TokenWindowDataset(token_ids[train_end:], BLOCK_SIZE, STRIDE)

print(f'Train windows: {len(train_ds):,}  |  Val windows: {len(val_ds):,}')
print(f'Block size: {BLOCK_SIZE}  |  Stride: {STRIDE}')

## 3 — Model Setup

Load `distilgpt2` and resize its embedding layer to match our custom vocabulary.

In [ ]:
model = GPT2LMHeadModel.from_pretrained('distilgpt2')
model.resize_token_embeddings(VOCAB_SIZE)

# Verify configuration
print(f'Model:        distilgpt2')
print(f'Parameters:   {sum(p.numel() for p in model.parameters()):,}')
print(f'Vocab resized: {model.config.vocab_size} → {VOCAB_SIZE}')
print(f'Context:      {model.config.n_positions}')
print(f'Layers:       {model.config.n_layer}')
print(f'Heads:        {model.config.n_head}')
print(f'Embed dim:    {model.config.n_embd}')

## 4 — Training

Cosine learning-rate schedule with warmup. Adjust `num_train_epochs` and
`per_device_train_batch_size` based on your hardware.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    overwrite_output_dir=True,
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

print('Training configuration:')
print(f'  Epochs:         {training_args.num_train_epochs}')
print(f'  Batch size:     {training_args.per_device_train_batch_size}')
print(f'  Grad accum:     {training_args.gradient_accumulation_steps}')
print(f'  Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print(f'  LR:             {training_args.learning_rate}')
print(f'  FP16:           {training_args.fp16}')

In [ ]:
# ── Train ────────────────────────────────────────────────────────────────
trainer.train()

# Save final model
model.save_pretrained(OUTPUT_DIR / 'final')
print(f'\nModel saved to {OUTPUT_DIR / "final"}')

## 5 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

log_hist = trainer.state.log_history
train_loss = [(e['step'], e['loss']) for e in log_hist if 'loss' in e]
eval_loss  = [(e['step'], e['eval_loss']) for e in log_hist if 'eval_loss' in e]

fig, ax = plt.subplots(figsize=(10, 4))
if train_loss:
    ax.plot(*zip(*train_loss), label='Train', alpha=0.7)
if eval_loss:
    ax.plot(*zip(*eval_loss), 'o-', label='Eval', markersize=4)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Training Loss — distilGPT-2 on 53-TET')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6 — Generation

Autoregressively generate token sequences conditioned on a style prompt.

In [ ]:
def generate_song(model, tok2id, id2tok, style: str = 'Jazz',
                  max_tokens: int = 300, temperature: float = 0.9,
                  top_k: int = 40) -> list[str]:
    """Generate a token sequence conditioned on <START> STYLE_x."""
    model.eval()
    device = next(model.parameters()).device

    prompt = [tok2id['<START>'], tok2id[f'STYLE_{style}']]
    input_ids = torch.tensor([prompt], dtype=torch.long, device=device)

    with torch.no_grad():
        out = model.generate(
            input_ids,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_k=top_k,
            do_sample=True,
            pad_token_id=tok2id.get('<END>', 0),
        )

    tokens = [id2tok.get(int(i), '?') for i in out[0]]
    # Truncate at first <END>
    if '<END>' in tokens:
        tokens = tokens[:tokens.index('<END>') + 1]
    return tokens


# ── Generate samples for different styles ────────────────────────────────
for style in ['Jazz', 'Bossa', 'Blues']:
    if f'STYLE_{style}' not in tok2id:
        print(f'STYLE_{style} not in vocabulary — skipping.')
        continue
    gen = generate_song(model, tok2id, id2tok, style=style)
    n_chords = gen.count('<CHORD_START>')
    print(f'\n── STYLE_{style}  ({n_chords} chords, {len(gen)} tokens) ──')
    print(' '.join(gen[:60]))
    if len(gen) > 60:
        print('...')

## 7 — Evaluate Generated Structure

Verify that generated sequences respect the token hierarchy.

In [ ]:
def validate_structure(tokens: list[str]) -> dict:
    """Check structural integrity of a generated token sequence."""
    checks = {
        'has_start': tokens[0] == '<START>' if tokens else False,
        'has_end':   tokens[-1] == '<END>' if tokens else False,
        'has_style': any(t.startswith('STYLE_') for t in tokens),
        'chord_balance': tokens.count('<CHORD_START>') == tokens.count('<CHORD_END>'),
        'midi_balance':  tokens.count('<MIDI_START>') == tokens.count('<MIDI_END>'),
        'n_chords': tokens.count('<CHORD_START>'),
    }
    checks['valid'] = all(checks[k] for k in ['has_start', 'has_end', 'has_style',
                                                'chord_balance', 'midi_balance'])
    return checks


# Validate the generated samples
for style in ['Jazz', 'Bossa', 'Blues']:
    if f'STYLE_{style}' not in tok2id:
        continue
    gen = generate_song(model, tok2id, id2tok, style=style)
    v = validate_structure(gen)
    status = '✓' if v['valid'] else '✗'
    print(f"{status} STYLE_{style}: {v['n_chords']} chords | "
          f"start={v['has_start']} end={v['has_end']} "
          f"chord_bal={v['chord_balance']} midi_bal={v['midi_balance']}")

---

## Notes

- Switch `TOKEN_DIR` from `test` to `full` for production training
- Increase `num_train_epochs` (20–50) for better convergence on large data
- Consider increasing `block_size` up to 1024 (distilgpt2 supports it)
- For multi-GPU: use `accelerate launch` or set `CUDA_VISIBLE_DEVICES`
- The model learns token structure (hierarchy) from data — no hardcoded grammar